In [1]:
import transformers

/home/athankar/miniconda3/envs/redteam/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch

In [3]:
a  = torch.zeros(2,3)

In [4]:
a.shape

torch.Size([2, 3])

In [5]:
a.unsqueeze(-1).shape

torch.Size([2, 3, 1])

In [6]:
a.unsqueeze(0).shape

torch.Size([1, 2, 3])

In [7]:
# config = transformers.AutoConfig.from_pretrained(
#         "mistralai/Mistral-7B-Instruct-v0.1",
#         trust_remote_code=True,
#         cache_dir="/data/tir/projects/tir6/bisk/athankar/projects/.cache",
#     )

model = transformers.AutoModelForCausalLM.from_pretrained(
        "mistralai/Mistral-7B-Instruct-v0.1",
        trust_remote_code=True,
        cache_dir="/data/tir/projects/tir6/bisk/athankar/projects/.cache",
    )

Loading checkpoint shards: 100%|███████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.64s/it]


In [8]:
out = model(**{
            "input_ids": torch.ones(size = (2, 4096)).long(),
            "labels": torch.ones(size = (2, 4096)).long(),
            "attention_mask": torch.ones(size = (2, 4096)).long(),
        })

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


In [82]:
model.parameters

<bound method Module.parameters of MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): Mistr

In [9]:
type(out)

transformers.modeling_outputs.CausalLMOutputWithPast

In [17]:
out["logits"].shape
# B x input_len x embed_dim

torch.Size([2, 4096, 32000])

In [20]:
import torch.nn as nn
log_probs = -nn.functional.log_softmax(out["logits"], dim=-1)

In [26]:
labels = torch.ones(size = (2, 4096,32000)).long()

In [27]:


# labels = torch.clamp(labels, min=0)
nll_loss = log_probs.gather(dim=-1, index=labels)

In [28]:
log_probs.shape # B x fixed_sequence_length x vocabulary_size

torch.Size([2, 4096, 32000])

In [29]:
labels.shape

torch.Size([2, 4096, 32000])

In [30]:
nll_loss.shape  # B x fixed_sequence_length x vocabulary_size


torch.Size([2, 4096, 32000])

In [31]:
log_probs.sum(dim=-1, keepdim=True, dtype=torch.float32).shape

torch.Size([2, 4096, 1])

In [32]:
nll_loss.shape

torch.Size([2, 4096, 32000])

In [41]:
pm = torch.tensor([1., 0.]).view(-1, 1, 1)

nll_loss *pm 

tensor([[[21.0105, 21.0105, 21.0105,  ..., 21.0105, 21.0105, 21.0105],
         [21.0105, 21.0105, 21.0105,  ..., 21.0105, 21.0105, 21.0105],
         [21.0105, 21.0105, 21.0105,  ..., 21.0105, 21.0105, 21.0105],
         ...,
         [21.0105, 21.0105, 21.0105,  ..., 21.0105, 21.0105, 21.0105],
         [21.0105, 21.0105, 21.0105,  ..., 21.0105, 21.0105, 21.0105],
         [21.0105, 21.0105, 21.0105,  ..., 21.0105, 21.0105, 21.0105]],

        [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         ...,
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]],
       grad_fn=<MulBackward0>)

In [35]:
nll_loss.shape

torch.Size([2, 4096, 32000])

In [36]:
torch.tensor([1., 0.]).shape

torch.Size([2])

In [37]:
pm = torch.tensor([1., 0.]).view(-1, 1, 1)


In [39]:
pm.shape

torch.Size([2, 1, 1])

In [73]:
from fastchat.model.model_adapter import get_conversation_template

def non_system_messages(raw_msg):
    conv = get_conversation_template("gpt-4")
    for msg in raw_msg:
        if msg["role"] == "system":
            continue
        conv.append_message(role=msg["role"], message=msg["content"])
    return conv.to_openai_api_messages()[1:]  # remove system prompt

In [83]:
c= torch.ones((3, 3))

In [85]:
c[0] = 0

In [86]:
c

tensor([[0., 0., 0.],
        [1., 1., 1.],
        [1., 1., 1.]])

In [87]:
torch.exp(c)

tensor([[1.0000, 1.0000, 1.0000],
        [2.7183, 2.7183, 2.7183],
        [2.7183, 2.7183, 2.7183]])